# providers.azure

OpenAI chat completions served from an Azure deployment.

In [ ]:
#| default_exp providers.azure

In [ ]:
#| hide
from nbdev.showdoc import *

## The provider

Construction is cheap and side-effect-free — only `_get_client` reaches for `AZURE_API_KEY`. That keeps `from nbdialog.providers.azure import AzureProvider` safe in environments without credentials (offline doc builds, CI without secrets, tests that mock `complete`).

Defaults match the constants that used to live in `core` so existing notebooks need exactly one line of setup to keep working:

```python
from nbdialog.providers.azure import AzureProvider
set_provider(AzureProvider())
```

In [ ]:
#| export
import os
from openai import AzureOpenAI

class AzureProvider:
    "OpenAI chat completions via an Azure deployment."
    def __init__(self,
                 deployment: str = "gpt-5.4",
                 endpoint: str = "https://pablo-ml1b1csr-eastus2.cognitiveservices.azure.com",
                 api_version: str = "2024-12-01-preview",
                 api_key_env: str = "AZURE_API_KEY",
                 max_completion_tokens: int = 16384):
        self.deployment, self.endpoint = deployment, endpoint
        self.api_version, self.api_key_env = api_version, api_key_env
        self.max_completion_tokens = max_completion_tokens
        self._client = None

    def _get_client(self):
        if self._client is None:
            self._client = AzureOpenAI(api_key=os.environ[self.api_key_env],
                                       azure_endpoint=self.endpoint,
                                       api_version=self.api_version)
        return self._client

    def complete(self, messages: list[dict]) -> str:
        resp = self._get_client().chat.completions.create(
            model=self.deployment, messages=messages,
            max_completion_tokens=self.max_completion_tokens,
        )
        return resp.choices[0].message.content

Smoke test — only runs when credentials are present, so the doc build stays green without them.

In [ ]:
if os.environ.get("AZURE_API_KEY"):
    print(AzureProvider().complete([{"role":"user","content":"ping"}]))

pong


## Build

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()